In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Outlier Detection

```{admonition} Goal
:class: tip
Detect and visualize outliers in sensor data using the Interquartile Range (IQR) method. Outliers can indicate sensor errors, unusual events (e.g. window openings), or data quality issues.
```

```{admonition} Data Basis
:class: note
**Dataset:** `flatTempHum.csv` — Hourly indoor temperature and humidity measurements from four residential flats (A–D).  
**Column used:** `FlatA_Temp`  
**Time range:** October 2018 – September 2020
```

## Data Preparation

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatTempHum.csv"
df = pd.read_csv(url, sep=";")
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time", drop=True)
df.head()

## Outlier Calculation

The IQR method flags values outside [Q1 − k×IQR, Q3 + k×IQR] as outliers, where k is typically 1.5.

In [ ]:
from pyedautils.data_quality import calc_outliers

result = calc_outliers(df, column="FlatA_Temp", multiplier=1.5)

print(f"IQR fences: [{result['lower']:.1f}, {result['upper']:.1f}] °C")
print(f"Outliers:   {result['count']} of {len(df)} ({result['percentage']}%)")

## Visualization

In [ ]:
from pyedautils.plots.stat import plot_outliers

fig = plot_outliers(
    df,
    column="FlatA_Temp",
    title="FlatA Temperature — Outlier Detection (IQR)",
    ylab="Temperature [°C]",
)
fig.show()

### Adjusting Sensitivity

The `multiplier` parameter controls how aggressive the detection is. A lower value flags more outliers.

In [ ]:
for k in [1.0, 1.5, 2.0, 3.0]:
    r = calc_outliers(df, column="FlatA_Temp", multiplier=k)
    print(f"k={k:.1f}  →  fences [{r['lower']:.1f}, {r['upper']:.1f}]  →  {r['count']:>4d} outliers ({r['percentage']}%)")

In [ ]:
fig = plot_outliers(
    df,
    column="FlatA_Temp",
    multiplier=3.0,
    title="FlatA Temperature — Outlier Detection (IQR × 3.0)",
    ylab="Temperature [°C]",
)
fig.show()

```{dropdown} Customization
- Use `multiplier=1.5` (default) for standard outlier detection, `3.0` for extreme outliers only.
- `calc_outliers()` returns a dict with `lower`, `upper`, `outliers` (DataFrame), `count`, and `percentage` — useful for downstream filtering.
- Combine with domain knowledge: a temperature reading of 15 °C indoors may be a real event (window open in winter), not a sensor error.
```